In [4]:
import tensorflow as tf
import numpy as np
import cv2

# Load your trained model

model = tf.keras.models.load_model("/Users/ektamishra/Desktop/dti_project_final copy/model/elite_retina_model.h5") # already defined

# Choose last conv layer
last_conv_layer_name = "top_conv"

# Create a model that maps input → last conv + predictions
grad_model = tf.keras.models.Model(
    [model.inputs],
    [model.get_layer(last_conv_layer_name).output, model.output]
)

In [5]:
def get_gradcam_heatmap(img_array, model, grad_model, class_index=None):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        loss = predictions[:, class_index]

    grads = tape.gradient(loss, conv_outputs)

    # Mean intensity of gradients over feature map
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [6]:
def overlay_heatmap(img, heatmap, alpha=0.4):
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0)

    return overlay

In [8]:
# Preprocess your image
img = preprocess_image(...)  # your pipeline
img_batch = np.expand_dims(img, axis=0)

# Prediction
preds = model.predict(img_batch)

# Grad-CAM
heatmap = get_gradcam_heatmap(img_batch, model, grad_model)

# Overlay
result = overlay_heatmap(img, heatmap)

# Show
import matplotlib.pyplot as plt
plt.imshow(result)
plt.axis('off')
plt.show()

NameError: name 'preprocess_image' is not defined